# 03. 인터랙티브 3D 뷰어와 마우스 모캡(Mocap)을 이용한 실시간 로봇 팔 제어

이 노트북은 `02_robot_arms.ipynb`의 확장 튜토리얼로(기존 `robot_arm_mocap.py`를 노트북 환경으로 통합·변환), **MuJoCo 내장 인터랙티브 3D GUI 뷰어(`mujoco.viewer`)**를 띄우고 마우스로 목표 타겟(Mocap)을 직접 움직이며 **작업 공간 임피던스 제어(Operational Space Control, OSC)**를 실시간으로 체험하는 방법을 다룹니다.

![kuka_mocap](./imgs/kuka_mocap.gif)

---
### 학습 목표
1. **작업 공간 임피던스 제어(OSC) 모듈화**: 위치(3D)와 자세(3D 회전)를 동시에 추종하는 토크 제어기를 함수로 구조화하기
2. **노트북 내 6자유도(위치+자세) 모캡 시뮬레이션**: 타겟의 3차원 위치와 자세가 동시에 변할 때 로봇 팔이 추종하는 모습 확인하기
3. **Windows 인터랙티브 3D 뷰어(`launch_passive`) 실습**: 마우스로 타겟을 실시간으로 끌고 회전시키거나, 로봇 몸체에 외력(외란)을 가해 임피던스 제어의 순응성(Compliance)을 직접 체험하기

In [ ]:
import time
import mujoco
import mujoco.viewer
import mediapy as media
import numpy as np

# Windows 환경 초기화 (내장 FFmpeg 자동 연결 및 프로젝트 루트 경로 설정)
from utils.notebook_setup import init_notebook
init_notebook()

## 1. KUKA iiwa 14 로봇 모델 및 작업 공간 제어기(OSC) 설정

7자유도 토크 제어 로봇인 **KUKA LBR iiwa 14** 모델을 불러오고, 매 제어 주기마다 목표 타겟(`mocap_id`)과 엔드이펙터(`site_id`) 사이의 6자유도 오차를 계산하여 관절 토크 $\tau$를 반환하는 제어기 클래스를 정의합니다.

In [ ]:
# KUKA iiwa 14 모델 불러오기
model = mujoco.MjModel.from_xml_path("robot/robot_arms/kuka_iiwa_14/scene.xml")
data = mujoco.MjData(model)

class OperationalSpaceController:
    """작업 공간 임피던스 제어(OSC) + 영공간(Null-space) 자세 제어기"""
    def __init__(self, model: mujoco.MjModel):
        self.model = model
        self.mocap_id = model.body("target").mocapid[0]
        self.site_id = model.site("attachment_site").id

        # 연산용 배열 사전 할당 (매 스텝 메모리 할당 오버헤드 방지)
        self.M_inv = np.zeros((model.nv, model.nv))
        self.jac = np.zeros((6, model.nv))
        self.error = np.zeros(6)
        self.error_pos = self.error[:3]
        self.error_ori = self.error[3:]
        self.site_quat = np.zeros(4)
        self.target_quat_conj = np.zeros(4)
        self.error_quat = np.zeros(4)

        # 작업 공간 임피던스 제어 게인 (Kp, Kd)
        impedance_pos = np.asarray([100.0, 100.0, 100.0])  # 위치 강성 [N/m]
        impedance_ori = np.asarray([50.0, 50.0, 50.0])     # 회전 강성 [Nm/rad]
        self.Kp = np.concatenate([impedance_pos, impedance_ori], axis=0)

        damping_ratio = 1.0
        damping_pos = damping_ratio * 2 * np.sqrt(impedance_pos)
        damping_ori = damping_ratio * 2 * np.sqrt(impedance_ori)
        self.Kd = np.concatenate([damping_pos, damping_ori], axis=0)

        # 영공간(Null-space) 초기 관절 자세 선호 게인
        self.q0 = model.key(0).qpos.copy()
        self.Kp_null = np.asarray([75.0, 75.0, 50.0, 50.0, 40.0, 25.0, 25.0])
        self.Kd_null = damping_ratio * 2 * np.sqrt(self.Kp_null)

    def compute_torque(self, data: mujoco.MjData) -> np.ndarray:
        """현재 상태에서 목표 모캡 타겟을 추종하기 위한 관절 토크 벡터(tau)를 계산합니다."""
        # 1. 위치 및 자세 오차 계산
        self.error_pos[:] = data.site(self.site_id).xpos - data.mocap_pos[self.mocap_id]
        target_ori = data.mocap_quat[self.mocap_id]
        mujoco.mju_negQuat(self.target_quat_conj, target_ori)
        mujoco.mju_mat2Quat(self.site_quat, data.site(self.site_id).xmat)
        mujoco.mju_mulQuat(self.error_quat, self.site_quat, self.target_quat_conj)
        mujoco.mju_quat2Vel(self.error_ori, self.error_quat, 1.0)

        # 2. 엔드이펙터 기하 자코비안 계산
        mujoco.mj_jacSite(self.model, data, self.jac[:3], self.jac[3:], self.site_id)

        # 3. 작업 공간 관성 행렬 Mx = (J * M^{-1} * J^T)^{-1} 계산
        mujoco.mj_solveM(self.model, data, self.M_inv, np.eye(self.model.nv))
        Mx_inv = self.jac @ self.M_inv @ self.jac.T
        Mx = np.linalg.pinv(Mx_inv)

        # 4. 작업 공간 임피던스 토크 + 중력/코리올리 보상 토크
        tau = self.jac.T @ Mx @ (-self.Kp * self.error - self.Kd * (self.jac @ data.qvel)) + data.qfrc_bias

        # 5. 영공간(Null-space) 관절 자세 안정화 토크 투영
        Jbar = self.M_inv @ self.jac.T @ Mx
        ddq = -self.Kp_null * (data.qpos - self.q0) - self.Kd_null * data.qvel
        tau += (np.eye(self.model.nv) - self.jac.T @ Jbar.T) @ ddq

        # 액추에이터 토크 한계 범위 내로 클리핑
        np.clip(tau, *self.model.actuator_ctrlrange.T, out=tau)
        return tau

controller = OperationalSpaceController(model)
print("OSC 제어기 초기화 완료!")

## 2. 실습 A: 노트북 내부에서 3차원 8자(Lissajous) 모캡 이동 및 회전 시뮬레이션

인터랙티브 창을 띄우기 전에, 마우스로 타겟을 입체적으로 움직이고 회전시키는 상황을 수식으로 모사하여 노트북 내부 비디오로 먼저 확인해 보겠습니다.
타겟의 $(x, y, z)$ 위치를 3차원 8자 궤적으로 움직이면서 동시에 목표 자세(`mocap_quat`)도 흔들어 주었을 때 KUKA 로봇 팔이 부드럽게 따라가는지 확인합니다:

In [ ]:
duration = 6.0  # (초)
framerate = 30  # (Hz)
frames = []

renderer = mujoco.Renderer(model, height=480, width=640)
scene_option = mujoco.MjvOption()
scene_option.frame = mujoco.mjtFrame.mjFRAME_SITE
scene_option.sitegroup[4] = 1
scene_option.geomgroup[4] = 1

# 초기 키프레임으로 리셋
mujoco.mj_resetDataKeyframe(model, data, 0)
mujoco.mj_forward(model, data)
init_mocap_pos = data.mocap_pos[controller.mocap_id].copy()
init_mocap_quat = data.mocap_quat[controller.mocap_id].copy()

while data.time < duration:
    t = data.time
    # 3차원 8자(Lissajous) 위치 궤적 생성
    data.mocap_pos[controller.mocap_id] = init_mocap_pos + np.array([
        0.10 * np.sin(2 * np.pi * 0.4 * t),
        0.15 * np.sin(2 * np.pi * 0.8 * t),
        0.08 * np.sin(2 * np.pi * 0.4 * t)
    ])
    # 타겟의 자세(쿼터니언)도 주기적으로 기울임
    angle = 0.35 * np.sin(2 * np.pi * 0.4 * t)
    axis = np.array([1.0, 0.5, 0.0])
    axis /= np.linalg.norm(axis)
    delta_quat = np.zeros(4)
    mujoco.mju_axisAngle2Quat(delta_quat, axis, angle)
    mujoco.mju_mulQuat(data.mocap_quat[controller.mocap_id], delta_quat, init_mocap_quat)

    # OSC 토크 계산 및 스텝 진행
    data.ctrl = controller.compute_torque(data)
    mujoco.mj_step(model, data)

    if len(frames) < data.time * framerate:
        renderer.update_scene(data, scene_option=scene_option)
        frames.append(renderer.render())

renderer.close()
media.show_video(frames, fps=framerate)

## 3. 실습 B: Windows 인터랙티브 3D 뷰어 실행 (`mujoco.viewer.launch_passive`)

이제 아래 셀을 실행하면 **별도의 MuJoCo 3D 인터랙티브 뷰어 창**이 열립니다!

### 마우스 및 키보드 조작 방법 (Windows)
1. **카메라 시점 조작**:
   - **회전(Rotate)**: 마우스 **왼쪽 버튼** 드래그
   - **이동(Pan)**: 마우스 **오른쪽 버튼** 드래그
   - **확대/축소(Zoom)**: 마우스 **휠 스크롤**
2. **목표 타겟(Mocap 상자) 실시간 조작**:
   - 로봇 손끝에 있는 반투명 타겟 상자를 **마우스 왼쪽 더블클릭**하여 선택합니다.
   - **`Ctrl` + 마우스 오른쪽 드래그**: 타겟의 **3차원 위치(Position)**를 마우스로 실시간 이동시킵니다.
   - **`Ctrl` + 마우스 왼쪽 드래그**: 타겟의 **3차원 자세(Orientation)**를 실시간 회전시킵니다.
3. **로봇 팔에 직접 외력(Perturbation) 가해보기**:
   - 로봇의 아무 관절 링크나 **더블클릭**한 뒤 **`Ctrl` + 마우스 드래그**로 당겨보세요!
   - 작업 공간 제어(OSC)와 영공간 임피던스 제어 덕분에 로봇이 용수철처럼 부드럽게 힘을 받아내며 다시 원래 포즈로 복원되는 **순응성(Compliance)**을 직접 확인할 수 있습니다.

> **안내:** 뷰어 창의 우측 상단 닫기(`X`) 버튼을 누르거나 `ESC` 키를 누르면 시뮬레이션 루프가 종료되고 노트북 셀 실행이 완료됩니다.

In [ ]:
# 이 셀을 실행하면 별도의 MuJoCo 3D 뷰어 창이 열립니다. (창을 닫으면 셀 실행이 종료됩니다)
with mujoco.viewer.launch_passive(model=model, data=data) as viewer:
    # 시뮬레이션을 초기 키프레임으로 리셋
    mujoco.mj_resetDataKeyframe(model, data, 0)
    # 자유 카메라 시점 초기화
    mujoco.mjv_defaultFreeCamera(model, viewer.cam)
    
    # 사이트 좌표축(frame) 및 모캡 타겟 그룹(group 4) 시각화 활성화
    viewer.opt.frame = mujoco.mjtFrame.mjFRAME_SITE
    viewer.opt.sitegroup[4] = 1
    viewer.opt.geomgroup[4] = 1

    print("인터랙티브 뷰어가 실행 중입니다. 뷰어 창에서 타겟을 더블클릭 후 Ctrl + 마우스 드래그로 움직여 보세요!")
    print("종료하려면 뷰어 창을 닫거나 ESC를 누르세요.")

    while viewer.is_running():
        step_start = time.time()

        # OSC 토크 계산 및 물리 스텝 진행
        data.ctrl = controller.compute_torque(data)
        mujoco.mj_step(model, data)

        # 뷰어 화면 동기화
        viewer.sync()

        # 실제 시간(Real-time) 속도에 맞춰 대기
        time_until_next_step = model.opt.timestep - (time.time() - step_start)
        if time_until_next_step > 0:
            time.sleep(time_until_next_step)

print("인터랙티브 뷰어가 정상적으로 종료되었습니다.")